# MOFSemNet — Google Colab setup

Run this once per Colab session, before importing anything from `src/`. Everything here is
Colab-specific (Drive mounting, matching torch-geometric's compiled extensions to Colab's
preinstalled CUDA build) and deliberately kept out of `src/` so the core library code stays
identical whether it's run here, on a personal GPU machine, or on a university server —
device selection itself is handled by `src.common.runtime.get_device`, not by anything below.

## 1. Mount Google Drive (for data/, outputs/, data/embeddings/ persistence across sessions)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Point this at wherever you keep the repo in Drive, e.g.:
PROJECT_DIR = '/content/drive/MyDrive/MOFSemNet'

## 2. Clone or pull the repo into that Drive path

In [ ]:
import os

if not os.path.isdir(PROJECT_DIR):
    !git clone <your-repo-url> "$PROJECT_DIR"
else:
    !cd "$PROJECT_DIR" && git pull

%cd $PROJECT_DIR

## 3. Install dependencies

Colab ships its own torch build (usually already CUDA-enabled) — check it before touching
torch/torch_geometric, since reinstalling torch on Colab is slow and sometimes breaks the
preinstalled CUDA linkage. `torch_geometric.nn.GCNConv/SAGEConv/GATConv` (the layers this
project uses) don't need the optional compiled extensions (torch-scatter/torch-sparse), so the
plain `pip install torch_geometric` below is normally sufficient even on Colab's GPU runtime.

In [ ]:
import torch
print('Colab torch version:', torch.__version__, '| CUDA available:', torch.cuda.is_available())

In [ ]:
# Installs everything EXCEPT torch (Colab's preinstalled build is kept as-is).
!grep -v '^torch==' requirements.txt > /tmp/requirements-colab.txt
!pip install -q -r /tmp/requirements-colab.txt

## 4. Verify device resolution picks up Colab's GPU

In [ ]:
from src.common.config import RuntimeConfig
from src.common.runtime import get_device

print(get_device(RuntimeConfig()))  # should print 'cuda' on a GPU runtime, 'cpu' otherwise